PLEASE REFER TO APPENDIX ON HOW TO RUN THE CODE, CHANGE THE FILE NAMES ACCORDINGLY IN ORDER FOR THE DATASETS TO BE IMPORTED ON YOUR SYSTEM. YOU CAN LOCATE THE WHEREABOUTS OF THE CSV'S BY CHECKING THE PATHWAY WHERE THE CODE IS TRYING TO IMPORT


USE localized_url function if there's an error with the imports

In [ ]:
import os
def localized_url(url):

    current_dir = os.getcwd().split("statistical-analysis-and-modeling-of-football-data")[0] + "statistical-analysis-and-modeling-of-football-data"
    branch = url.split("statistical-analysis-and-modeling-of-football-data")[1]
    return current_dir + branch

In [2]:
import numpy as np
import pandas as pd
import re
import time
import requests
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium_stealth import stealth
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import logging
import functools


In [3]:
url="https://www.fifaindex.com/teams/fifa19/?league=13&order=desc"
response = requests.get(url)
if response.status_code == 200:
    print("Successfully fetched the page")
else:
    print(f"Failed to fetch the page, status code: {response.status_code}")

Failed to fetch the page, status code: 403


In [20]:

# Clear existing handlers
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)



In [3]:

def retry(max_attempts=3, delay=3, exceptions=(Exception,), logger=None):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            attempts = 0
            while attempts < max_attempts:
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    attempts += 1
                    if logger:
                        logger.warning(f"Attempt {attempts} failed for {func.__name__}: {e}")
                    else:
                        print(f"Attempt {attempts} failed for {func.__name__}: {e}")
                    if attempts == max_attempts:
                        if logger:
                            logger.error(f"{func.__name__} failed after {max_attempts} attempts.")
                        raise
                    time.sleep(delay)
        return wrapper
    return decorator

In [4]:
def initialize_driver():
    options = webdriver.ChromeOptions()
    options.add_argument('--headless')
    options.add_argument("--start-maximized")


    driver = webdriver.Chrome(options=options)

    stealth(driver,
            languages=["en-US", "en"],
            vendor="Google Inc.",
            platform="Win32",
            webgl_vendor="Intel Inc.",
            renderer="Intel Iris OpenGL Engine",
            fix_hairline=True)
    
    return driver

In [5]:
@retry(max_attempts=3, delay=5, exceptions=(Exception,), logger=logging)
def scrape_sofifa_player_stats(url, team_name, version):
    driver = initialize_driver()
    driver.get(url)

    # Accept cookies via shadow DOM
    try:
        shadow_host = driver.find_element(By.CSS_SELECTOR, "#cmpwrapper")
        shadow_root = driver.execute_script("return arguments[0].shadowRoot", shadow_host)
        accept_button = shadow_root.find_element(By.CSS_SELECTOR, "#cmpwelcomebtnyes")
        accept_button.click()
        time.sleep(1)
    except Exception as e:
        logging.warning(f"Could not interact with cookie popup on {url}: {e}")
    

    # Wait for stat cards
    WebDriverWait(driver, 10).until(
        EC.presence_of_all_elements_located((By.CLASS_NAME, "card"))
    )

    desired_categories = {
        "Ball Skills", "Passing", "Shooting", "Defence",
        "Physical", "Mental", "Goalkeeper"
    }

    player_stats = {}
    cards = driver.find_elements(By.CLASS_NAME, "card")

    for card in cards:
        try:
            category = card.find_element(By.CLASS_NAME, "card-header").text.strip()
            if category not in desired_categories:
                continue
            body = card.find_element(By.CLASS_NAME, "card-body")
            p_tags = body.find_elements(By.TAG_NAME, "p")
            for p in p_tags:
                stat_name = p.text.split('\n')[0].strip()
                stat_value = p.find_element(By.CLASS_NAME, "float-right").text.strip()
                player_stats[stat_name] = stat_value
        except Exception as e:
            logging.warning(f"Skipping a stat card due to error: {e}")
            continue

    df = pd.DataFrame(player_stats, columns=player_stats.keys(), index=[0])
    df['Team'] = team_name
    df['Version'] = version
    driver.quit()
    return df
    


In [6]:
@retry(max_attempts=3, delay=5, exceptions=(Exception,), logger=logging)
def get_team_links_df(url,version):
    driver = initialize_driver()

    driver.get(url)

    # Accept cookies via shadow DOM
    try:
        shadow_host = driver.find_element(By.CSS_SELECTOR, "#cmpwrapper")
        shadow_root = driver.execute_script("return arguments[0].shadowRoot", shadow_host)
        accept_button = shadow_root.find_element(By.CSS_SELECTOR, "#cmpwelcomebtnyes")
        accept_button.click()
        time.sleep(1)
    except Exception as e:
        logging.warning(f"Could not interact with cookie popup on {url}: {e}")

    # Scrape team links and names
    team_links = driver.find_elements(By.CSS_SELECTOR, "table.table-teams a.link-team")
    team_data = set()
    for link in team_links:
        name = link.text.strip()
        href = link.get_attribute("href")
        if name and href:
            team_data.add((name, href))  # Use set to avoid duplicates



    # Convert to DataFrame
    df = pd.DataFrame(sorted(team_data), columns=["Team Name", "Team URL"])
    df["Version"] = version
    driver.quit()
    logging.info(f"Extracted {len(df)} teams from {url} with version {df['Version'].iloc[0]}")
    return df


In [7]:
@retry(max_attempts=3, delay=5, exceptions=(Exception,), logger=logging)
def get_player_links(url, team_name):
    # Setup driver
    driver = initialize_driver()

    driver.get(url)

    # Accept cookies via shadow DOM
    try:
        shadow_host = driver.find_element(By.CSS_SELECTOR, "#cmpwrapper")
        shadow_root = driver.execute_script("return arguments[0].shadowRoot", shadow_host)
        accept_button = shadow_root.find_element(By.CSS_SELECTOR, "#cmpwelcomebtnyes")
        accept_button.click()
        time.sleep(1)
    except Exception as e:
        logging.warning(f"Could not interact with cookie popup on {url}: {e}")
        logging.warning("Retrying with cookies visa shadow DOM interaction")
        time.sleep(5)
        try:
            shadow_host = driver.find_element(By.CSS_SELECTOR, "#cmpwrapper")
            shadow_root = driver.execute_script("return arguments[0].shadowRoot", shadow_host)
            accept_button = shadow_root.find_element(By.CSS_SELECTOR, "#cmpwelcomebtnyes")
            accept_button.click()
            time.sleep(1)
        except Exception as e:
            logging.error(f"Failed to accept cookies after retrying: {e}")
            logging.error("Retrying without cookie acceptance")
            try:
                accept_button = driver.find_element(By.CSS_SELECTOR, "#cmpwelcomebtnyes")
                accept_button.click()
                time.sleep(1)
            except Exception as e:
                logging.error(f"Failed to accept cookies without shadow DOM: {e}")
                

    # Extract data early to avoid stale references
    elements = driver.find_elements(By.CSS_SELECTOR, "table.table-players a.link-player")
    player_data = {
        (el.text.strip(), el.get_attribute("href")) for el in elements
        if el.text.strip() and el.get_attribute("href")
    }

    df = pd.DataFrame(sorted(player_data), columns=["Player Name", "Player URL"])
    df["Team Name"] = team_name
    df["Version"] = url.rstrip('/').split('/')[-1]
    driver.quit()
    logging.info(f"Extracted {len(df)} players from {team_name} with version {df['Version'].iloc[0]}")
    return df
    


In [8]:
@retry(max_attempts=3, delay=5, exceptions=(Exception,), logger=logging)
def get_player_stats_from_team(team_df):
    player_stats = pd.DataFrame()

    for index, row in team_df.iterrows():
        player_url = row['Player URL']
        player_name = row['Player Name']
        logging.info(f"Scraping stats for {player_name} from {player_url}...")

        try:
            stats_df = scrape_sofifa_player_stats(player_url, row["Team Name"], row["Version"])  # Should return a DataFrame
            stats_df.insert(0, 'Player Name', player_name)     # Add name as first column
            player_stats = pd.concat([player_stats, stats_df], ignore_index=True)
            

        except Exception as e:
            logging.error(f"Error scraping {player_name}: {e}")
            logging.error("Retrying with a new driver instance")
            try:
                stats_df = scrape_sofifa_player_stats(player_url, row["Team Name"], row["Version"])  # Should return a DataFrame
                stats_df.insert(0, 'Player Name', player_name)     # Add name as first column
                player_stats = pd.concat([player_stats, stats_df], ignore_index=True)
            except Exception as e:
                logging.error(f"Failed to scrape stats for {player_name} after retry: {e}")
                continue
        else:
            logging.info(f"Successfully scraped stats for {player_name} from  team {row['Team Name']} , version {row['Version']}")
    
        

    return player_stats

#Lets create a big data frame with all players from all teams from all seasons
# To create a progress bar, we can use tqdm
from tqdm import tqdm
fifa_years_list = [i for i in range(16, 25)]
season_urls_list=["https://www.fifaindex.com/teams/fifa"+str(i)+"/?league=13&order=desc" for i in fifa_years_list]

all_teams_df = pd.DataFrame()
for url in tqdm(season_urls_list, desc="Gathering team links"):
    teams_df = get_team_links_df(url)
    all_teams_df = pd.concat([all_teams_df, teams_df], ignore_index=True)
# -- Step 2: Get all teams
all_teams_df = pd.DataFrame()
for url in tqdm(season_urls_list, desc="Gathering team links"):
    teams_df = get_team_links_df(url)
    all_teams_df = pd.concat([all_teams_df, teams_df], ignore_index=True)

# Now we have all players from all teams from all seasons, we can scrape their stats
# We can use the get_player_stats_from_team function to scrape the stats for each team

all_player_stats_df = pd.DataFrame()
for index, row in tqdm(all_players_df.iterrows(), total=len(all_players_df), desc="Gathering player stats"):
    team_name = row['Team Name']
    version = row['Version']
    team_stats_df = get_player_stats_from_team(all_players_df[all_players_df['Team Name'] == team_name])
    all_player_stats_df = pd.concat([all_player_stats_df, team_stats_df], ignore_index=True)
    


In [35]:
import pandas as pd
import time
import logging
from tqdm import tqdm
from pathlib import Path

# Setup logging
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    filename='scraper.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    filemode='a'
)
logging.info("--------------------------Setting up logging---------------------------")
# -- Step 1: Define URLs
fifa_years_dict = {}

for i in range(16, 25):
    fifa_years_dict[i] = f"https://www.fifaindex.com/teams/fifa{i}/?league=13&order=desc"
fifa_years_list = list(fifa_years_dict.keys())

# -- Step 2: Get all teams
all_teams_df = pd.DataFrame()
for i in tqdm(fifa_years_list, desc="Gathering team links"):
    url = fifa_years_dict[i]
    try:
        teams_df = get_team_links_df(url, f"fifa{i}")
        all_teams_df = pd.concat([all_teams_df, teams_df], ignore_index=True)
        time.sleep(1.5)
    except Exception as e:
        logging.error(f"❌ Error gathering teams for FIFA {i}: {e}")
        time.sleep(5)
        try:
            teams_df = get_team_links_df(url, f"fifa{i}")
            all_teams_df = pd.concat([all_teams_df, teams_df], ignore_index=True)
            time.sleep(1.5)
            logging.info(f"✅ Retry successful for FIFA {i}")
        except Exception as e2:
            logging.error(f"❌ Retry failed for FIFA {i}: {e2}")
            continue

# -- Step 3: Get all players
all_players_df = pd.DataFrame()
for index, row in tqdm(all_teams_df.iterrows(), total=len(all_teams_df), desc="Gathering player links"):
    team_url = row['Team URL']
    team_name = row['Team Name']
    try:
        players_df = get_player_links(team_url, team_name)
        all_players_df = pd.concat([all_players_df, players_df], ignore_index=True)
    except Exception as e:
        logging.error(f"Error gathering players for {team_name}: {e}")

# Save intermediate checkpoint
all_players_df.to_csv("all_players_checkpoint.csv", index=False)

# -- Step 4: Scrape Player Stats with Team-Level Efficiency

# Load progress if exists
checkpoint_path = Path("all_player_stats_checkpoint.csv")
if checkpoint_path.exists():
    all_player_stats_df = pd.read_csv(checkpoint_path)
    scraped_teams = set(zip(all_player_stats_df['Team'], all_player_stats_df['Version']))
else:
    all_player_stats_df = pd.DataFrame()
    scraped_teams = set()

# Group players by (team_name, version)
grouped_teams = all_players_df.groupby(['Team Name', 'Version'])

# Scrape each team only once
for (team_name, version), group in tqdm(grouped_teams, desc="Scraping team stats"):
    if (team_name, version) in scraped_teams:
        continue

    try:
        stats_df = get_player_stats_from_team(group)
        all_player_stats_df = pd.concat([all_player_stats_df, stats_df], ignore_index=True)

        # Save progress
        all_player_stats_df.to_csv("all_player_stats_checkpoint.csv", index=False)
        logging.info(f"✅ Scraped stats for {team_name} ({version})")

    except Exception as e:
        logging.error(f"❌ Failed scraping for {team_name} ({version}): {e}")
        continue

# Final save
all_player_stats_df.to_csv("all_player_stats_final.csv", index=False)
logging.info("✅ Scraping completed and final dataset saved.")


Scraping team stats: 100%|██████████| 180/180 [10:11:54<00:00, 203.97s/it] 


In [85]:
all_player_stats_df = pd.read_csv("all_player_stats_final.csv")

In [86]:
all_player_stats_df

,Player Name,Ball Control,Dribbling,Marking,Slide Tackle,Stand Tackle,Aggression,Reactions,Att. Position,Interceptions,...,Volleys,GK Positioning,GK Diving,GK Handling,GK Kicking,GK Reflexes,Team,Version,Stamina 6,Sprint Speed 1
0,Adam Smith,74.0,68.0,NaN,74.0,75.0,73.0,73.0,65.0,69.0,...,43.0,9.0,8.0,9.0,15.0,6.0,AFC Bournemouth,afc-bournemouth,NaN,NaN
1,Antoine Semenyo,69.0,72.0,NaN,11.0,19.0,62.0,68.0,68.0,14.0,...,56.0,15.0,10.0,11.0,14.0,7.0,AFC Bournemouth,afc-bournemouth,NaN,NaN
2,Chris Mepham,64.0,58.0,NaN,72.0,74.0,73.0,68.0,39.0,71.0,...,33.0,14.0,10.0,10.0,12.0,12.0,AFC Bournemouth,afc-bournemouth,NaN,NaN
3,Dango Ouattara,74.0,78.0,NaN,44.0,51.0,46.0,66.0,74.0,52.0,...,49.0,14.0,9.0,6.0,13.0,10.0,AFC Bournemouth,afc-bournemouth,NaN,NaN
4,Darren Randolph,33.0,14.0,NaN,14.0,14.0,29.0,70.0,11.0,18.0,...,16.0,70.0,74.0,70.0,66.0,73.0,AFC Bournemouth,afc-bournemouth,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7387,Ryan Giles,67.0,68.0,NaN,62.0,66.0,55.0,62.0,50.0,56.0,...,41.0,8.0,14.0,6.0,5.0,8.0,Wolves,fifa23,NaN,NaN
7388,Rúben Neves,82.0,75.0,NaN,77.0,79.0,82.0,81.0,62.0,80.0,...,82.0,8.0,16.0,12.0,10.0,10.0,Wolves,fifa23,NaN,NaN
7389,Saša Kalajdžić,78.0,74.0,NaN,47.0,46.0,44.0,78.0,81.0,34.0,...,73.0,14.0,7.0,6.0,13.0,15.0,Wolves,fifa23,NaN,NaN
7390,Theo Corbeanu,66.0,71.0,NaN,45.0,49.0,40.0,57.0,61.0,39.0,...,50.0,10.0,7.0,6.0,9.0,7.0,Wolves,fifa23,NaN,NaN


In [9]:
def nan_analysis(df):
    """
    This function takes a DataFrame and returns a DataFrame with the percentage of NaN values for each column.
    Only columns with NaN values are included in the output DataFrame.
    """
    nan_percentage = df.isna().mean() * 100
    nan_number = df.isna().sum()
    nan_percentage = nan_percentage[nan_percentage > 0].reset_index()
    nan_number = nan_number[nan_number > 0].reset_index()
    nan_percentage.columns = ['Column', 'Percentage of NaN Values']
    nan_number.columns = ['Column', 'Number of NaN Values']
    nan_percentage = pd.merge(nan_percentage, nan_number, on='Column')
    nan_percentage = nan_percentage.sort_values(by='Percentage of NaN Values', ascending=False)
    return nan_percentage

In [88]:
nan_analysis(all_player_stats_df)

,Column,Percentage of NaN Values,Number of NaN Values
35,Sprint Speed 1,99.986472,7391
34,Stamina 6,99.986472,7391
2,Marking,21.415043,1583
10,Composure,11.295996,835
22,Shot Power,0.798160,59
23,Finishing,0.798160,59
24,Long Shots,0.798160,59
25,Curve,0.798160,59
26,FK Acc.,0.798160,59
27,Penalties,0.798160,59


In [89]:
all_player_stats_df = all_player_stats_df.drop(columns=['Sprint Speed 1', 'Stamina 6'])

In [90]:
# Get the players who have nan values in their stats
players_with_nan = all_player_stats_df[all_player_stats_df.isna().any(axis=1)]
players_with_nan 

,Player Name,Ball Control,Dribbling,Marking,Slide Tackle,Stand Tackle,Aggression,Reactions,Att. Position,Interceptions,...,FK Acc.,Penalties,Volleys,GK Positioning,GK Diving,GK Handling,GK Kicking,GK Reflexes,Team,Version
0,Adam Smith,74.0,68.0,NaN,74.0,75.0,73.0,73.0,65.0,69.0,...,66.0,55.0,43.0,9.0,8.0,9.0,15.0,6.0,AFC Bournemouth,afc-bournemouth
1,Antoine Semenyo,69.0,72.0,NaN,11.0,19.0,62.0,68.0,68.0,14.0,...,39.0,51.0,56.0,15.0,10.0,11.0,14.0,7.0,AFC Bournemouth,afc-bournemouth
2,Chris Mepham,64.0,58.0,NaN,72.0,74.0,73.0,68.0,39.0,71.0,...,35.0,44.0,33.0,14.0,10.0,10.0,12.0,12.0,AFC Bournemouth,afc-bournemouth
3,Dango Ouattara,74.0,78.0,NaN,44.0,51.0,46.0,66.0,74.0,52.0,...,55.0,66.0,49.0,14.0,9.0,6.0,13.0,10.0,AFC Bournemouth,afc-bournemouth
4,Darren Randolph,33.0,14.0,NaN,14.0,14.0,29.0,70.0,11.0,18.0,...,14.0,30.0,16.0,70.0,74.0,70.0,66.0,73.0,AFC Bournemouth,afc-bournemouth
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7387,Ryan Giles,67.0,68.0,NaN,62.0,66.0,55.0,62.0,50.0,56.0,...,67.0,49.0,41.0,8.0,14.0,6.0,5.0,8.0,Wolves,fifa23
7388,Rúben Neves,82.0,75.0,NaN,77.0,79.0,82.0,81.0,62.0,80.0,...,81.0,76.0,82.0,8.0,16.0,12.0,10.0,10.0,Wolves,fifa23
7389,Saša Kalajdžić,78.0,74.0,NaN,47.0,46.0,44.0,78.0,81.0,34.0,...,47.0,57.0,73.0,14.0,7.0,6.0,13.0,15.0,Wolves,fifa23
7390,Theo Corbeanu,66.0,71.0,NaN,45.0,49.0,40.0,57.0,61.0,39.0,...,64.0,55.0,50.0,10.0,7.0,6.0,9.0,7.0,Wolves,fifa23


In [91]:
# Show all rows that doesn't have "fifa" in the Version column
all_player_stats_df[~all_player_stats_df['Version'].str.contains("fifa", case=False, na=False)]["Version"]
# Change these to fifa24
all_player_stats_df.loc[~all_player_stats_df['Version'].str.contains("fifa", case=False, na=False), 'Version'] = "fifa24"

In [92]:
# Same procedure for all_players_df
all_players_df.loc[~all_players_df['Version'].str.contains("fifa", case=False, na=False), 'Version'] = "fifa24"

In [93]:
all_players_df.columns

Index(['Player Name', 'Player URL', 'Team Name', 'Version'], dtype='object')

In [94]:
# The team name column in all_player_stats_df is not consistent with all_players_df
# Let's rename the 'Team' column in all_player_stats_df to 'Team Name'
all_player_stats_df.rename(columns={'Team': 'Team Name'}, inplace=True)
all_player_stats_df.columns

Index(['Player Name', 'Ball Control', 'Dribbling', 'Marking', 'Slide Tackle',
       'Stand Tackle', 'Aggression', 'Reactions', 'Att. Position',
       'Interceptions', 'Vision', 'Composure', 'Crossing', 'Short Pass',
       'Long Pass', 'Acceleration', 'Stamina', 'Strength', 'Balance',
       'Sprint Speed', 'Agility', 'Jumping', 'Heading', 'Shot Power',
       'Finishing', 'Long Shots', 'Curve', 'FK Acc.', 'Penalties', 'Volleys',
       'GK Positioning', 'GK Diving', 'GK Handling', 'GK Kicking',
       'GK Reflexes', 'Team Name', 'Version'],
      dtype='object')

In [95]:
#Now lets merge the two DataFrames on team name, version and player name so that we can get the url for each player
merged_with_url = pd.merge(all_player_stats_df, all_players_df, 
                               on=['Player Name', 'Team Name', 'Version'],
                               how='left')

In [96]:
players_with_nan=nan_analysis(merged_with_url)

In [97]:
merged_with_url

,Player Name,Ball Control,Dribbling,Marking,Slide Tackle,Stand Tackle,Aggression,Reactions,Att. Position,Interceptions,...,Penalties,Volleys,GK Positioning,GK Diving,GK Handling,GK Kicking,GK Reflexes,Team Name,Version,Player URL
0,Adam Smith,74.0,68.0,NaN,74.0,75.0,73.0,73.0,65.0,69.0,...,55.0,43.0,9.0,8.0,9.0,15.0,6.0,AFC Bournemouth,fifa24,https://www.fifaindex.com/player/190885/adam-s...
1,Antoine Semenyo,69.0,72.0,NaN,11.0,19.0,62.0,68.0,68.0,14.0,...,51.0,56.0,15.0,10.0,11.0,14.0,7.0,AFC Bournemouth,fifa24,https://www.fifaindex.com/player/241236/antoin...
2,Chris Mepham,64.0,58.0,NaN,72.0,74.0,73.0,68.0,39.0,71.0,...,44.0,33.0,14.0,10.0,10.0,12.0,12.0,AFC Bournemouth,fifa24,https://www.fifaindex.com/player/233201/chris-...
3,Dango Ouattara,74.0,78.0,NaN,44.0,51.0,46.0,66.0,74.0,52.0,...,66.0,49.0,14.0,9.0,6.0,13.0,10.0,AFC Bournemouth,fifa24,https://www.fifaindex.com/player/265552/dango-...
4,Darren Randolph,33.0,14.0,NaN,14.0,14.0,29.0,70.0,11.0,18.0,...,30.0,16.0,70.0,74.0,70.0,66.0,73.0,AFC Bournemouth,fifa24,https://www.fifaindex.com/player/169697/darren...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7387,Ryan Giles,67.0,68.0,NaN,62.0,66.0,55.0,62.0,50.0,56.0,...,49.0,41.0,8.0,14.0,6.0,5.0,8.0,Wolves,fifa23,https://www.fifaindex.com/player/243608/ryan-g...
7388,Rúben Neves,82.0,75.0,NaN,77.0,79.0,82.0,81.0,62.0,80.0,...,76.0,82.0,8.0,16.0,12.0,10.0,10.0,Wolves,fifa23,https://www.fifaindex.com/player/224293/r%C3%B...
7389,Saša Kalajdžić,78.0,74.0,NaN,47.0,46.0,44.0,78.0,81.0,34.0,...,57.0,73.0,14.0,7.0,6.0,13.0,15.0,Wolves,fifa23,https://www.fifaindex.com/player/239901/sa%C5%...
7390,Theo Corbeanu,66.0,71.0,NaN,45.0,49.0,40.0,57.0,61.0,39.0,...,55.0,50.0,10.0,7.0,6.0,9.0,7.0,Wolves,fifa23,https://www.fifaindex.com/player/258758/theo-c...


In [101]:
# Now that we have the corresponding urls to each player that have nan values in their stats, we can rerun the scraping function for those players
# Step 1: Identify players with missing values
# Get list of columns with NaN values
nan_columns = players_with_nan['Column'].tolist()

# Identify rows (players) where any of those columns are NaN
nan_players = merged_with_url[merged_with_url['Player URL'].notna() & merged_with_url['Player Name'].notna()]
nan_players = nan_players[nan_players[nan_columns].isna().any(axis=1)]

# Step 2: Re-scrape those players
updated_stats = pd.DataFrame()

for _, row in nan_players.iterrows():
    player_url = row['Player URL']
    player_name = row['Player Name']
    version = row['Version']
    team_name = row['Team Name']
    print(f"🔄 Rescraping: {player_name}")
    try:
        df = scrape_sofifa_player_stats(player_url, team_name, version)
        df['Player Name'] = player_name
        df['Team Name'] = row['Team Name']
        df['Version'] = row['Version']
        updated_stats = pd.concat([updated_stats, df], ignore_index=True)
    except Exception as e:
        print(f"❌ Error for {player_name}: {e}")
        continue


🔄 Rescraping: Adam Smith
🔄 Rescraping: Antoine Semenyo
🔄 Rescraping: Chris Mepham
🔄 Rescraping: Dango Ouattara
🔄 Rescraping: Darren Randolph
🔄 Rescraping: David Brooks
🔄 Rescraping: Dominic Sadi
🔄 Rescraping: Dominic Solanke
🔄 Rescraping: Emiliano Marcondes
🔄 Rescraping: Gavin Kilkenny
🔄 Rescraping: Hamed Traorè
🔄 Rescraping: Illia Zabarnyi
🔄 Rescraping: Ionuț Andrei RaduL
🔄 Rescraping: Jaidon Anthony
🔄 Rescraping: Jamal Lowe
🔄 Rescraping: James Hill
🔄 Rescraping: Joe Rothwell
🔄 Rescraping: Justin Kluivert
🔄 Rescraping: Kieffer Moore
🔄 Rescraping: Lewis Cook
🔄 Rescraping: Lloyd Kelly
🔄 Rescraping: Marcos Senesi
🔄 Rescraping: Marcus Tavernier
🔄 Rescraping: Mark Travers
🔄 Rescraping: Milos Kerkez
🔄 Rescraping: Nathan Moriah-Welsh
🔄 Rescraping: Neto
🔄 Rescraping: Owen Bevan
🔄 Rescraping: Philip Billing
🔄 Rescraping: Romain Faivre
🔄 Rescraping: Ryan Christie
🔄 Rescraping: Ryan Fredericks
🔄 Rescraping: Will Dennis
🔄 Rescraping: Adam Smith
🔄 Rescraping: Antoine Semenyo
🔄 Rescraping: Ben Pear

In [103]:
nan_analysis(updated_stats)

,Column,Percentage of NaN Values,Number of NaN Values
35,Sprint Speed 1,99.958695,2420
34,Stamina 6,99.958695,2420
10,Composure,33.292028,806
18,Sprint Speed,0.247831,6
20,Jumping,0.206526,5
26,FK Acc.,0.206526,5
21,Heading,0.206526,5
24,Long Shots,0.206526,5
19,Agility,0.206526,5
25,Curve,0.206526,5


In [105]:
updated_stats.to_csv("updated_player_stats.csv", index=False)

In [106]:
updated_stats.drop(columns=['Sprint Speed 1', "Stamina 6", "Composure"], inplace=True)

In [123]:
# Set index for clean update logic
#merged_with_url.set_index(['Player Name', 'Team Name', 'Version'], inplace=True)
#updated_stats.set_index(['Player Name', 'Team Name', 'Version'], inplace=True)

# Update only NaN values
for col in updated_stats.columns:
    merged_with_url[col] = merged_with_url[col].combine_first(updated_stats[col])

# Reset index back
merged_with_url.reset_index(inplace=True)


In [127]:
merged_with_url.to_csv("final_player_stats.csv", index=False)

In [10]:
jeremy_doku_link = "https://www.fifaindex.com/player/246420/j%C3%A9r%C3%A9my-doku/fifa24_593/"
#scrape jeremy_doku_link
#Team is Manchester City
#fifa24
jeremy_doku_stats = scrape_sofifa_player_stats(jeremy_doku_link, "Manchester City", "fifa24")
jeremy_doku_stats.insert(0, 'Player Name', 'Jérémy Doku')  # Add name as first column



In [11]:
jeremy_doku_stats

,Player Name,Ball Control 1,Dribbling 2,Marking,Slide Tackle,Stand Tackle,Aggression,Reactions 4,Att. Position,Interceptions,...,FK Acc.,Penalties,Volleys 5,GK Positioning,GK Diving,GK Handling,GK Kicking,GK Reflexes,Team,Version
0,Jérémy Doku,77,85,None,25,28,58,74,72,18,...,44,51,59,9,11,10,8,12,Manchester City,fifa24


In [14]:
jeremy_doku_stats.columns

Index(['Player Name', 'Ball Control 1', 'Dribbling 2', 'Marking',
       'Slide Tackle', 'Stand Tackle', 'Aggression', 'Reactions 4',
       'Att. Position', 'Interceptions', 'Vision', 'Composure 1', 'Crossing 2',
       'Short Pass 3', 'Long Pass 2', 'Acceleration', 'Stamina 2',
       'Strength 1', 'Balance', 'Sprint Speed', 'Agility 3', 'Jumping 6',
       'Heading', 'Shot Power 1', 'Finishing 3', 'Long Shots', 'Curve 3',
       'FK Acc.', 'Penalties', 'Volleys 5', 'GK Positioning', 'GK Diving',
       'GK Handling', 'GK Kicking', 'GK Reflexes', 'Team', 'Version'],
      dtype='object')

In [18]:
"""
['Player Name', 'Ball Control 1', 'Dribbling 2', 'Marking',
       'Slide Tackle', 'Stand Tackle', 'Aggression', 'Reactions 4',
       'Att. Position', 'Interceptions', 'Vision', 'Composure 1', 'Crossing 2',
       'Short Pass 3', 'Long Pass 2', 'Acceleration', 'Stamina 2',
       'Strength 1', 'Balance', 'Sprint Speed', 'Agility 3', 'Jumping 6',
       'Heading', 'Shot Power 1', 'Finishing 3', 'Long Shots', 'Curve 3',
       'FK Acc.', 'Penalties', 'Volleys 5', 'GK Positioning', 'GK Diving',
       'GK Handling', 'GK Kicking', 'GK Reflexes', 'Team', 'Version']
"""
# Rename columns to match the format of merged_with_url, which dont have the numbers at the end
jeremy_doku_stats.rename(columns={
    'Ball Control 1': 'Ball Control',
    'Dribbling 2': 'Dribbling',
    'Reactions 4': 'Reactions',
    'Short Pass 3': 'Short Pass',
    'Long Pass 2': 'Long Pass',
    'Composure 1': 'Composure',
    'Crossing 2': 'Crossing',
    'Stamina 2': 'Stamina',
    'Strength 1': 'Strength',
    'Agility 3': 'Agility',
    'Jumping 6': 'Jumping',
    'Shot Power 1': 'Shot Power',
    'Finishing 3': 'Finishing',
    'Curve 3': 'Curve',
    'Volleys 5': 'Volleys',
    'Team':'Team Name'}, inplace=True)
                

In [ ]:
merged_with_url = pd.read_csv("Data/final_player_stats.csv")
merged_with_url = pd.concat([merged_with_url, jeremy_doku_stats], ignore_index=True)


In [20]:
merged_with_url = pd.read_csv("Data/final_player_stats.csv")


In [21]:
#add jeremy doku url to the merged_with_url DataFrame
merged_with_url.loc[merged_with_url['Player Name'] == 'Jérémy Doku', 'Player URL'] = jeremy_doku_link

In [23]:
merged_with_url.to_csv("Data/final_player_stats.csv", index=False)